# 📚 Agentic RAG (Retrieval-Augmented Generation) with C#

## 📋 Overview

This notebook demonstrates how to build an intelligent agent that uses **Retrieval-Augmented Generation (RAG)** to answer questions based on uploaded documents. The agent uses Azure AI Foundry's vector store and file search capabilities to retrieve relevant information and generate accurate responses.

**Key Features:**
- 📁 **File Upload & Processing**: Upload documents to Azure AI Foundry
- 🔍 **Vector Store Creation**: Create searchable vector embeddings from documents
- 🤖 **RAG Agent**: Agent that retrieves and uses document context to answer questions
- 🎯 **Grounded Responses**: Answers based only on uploaded document content
- 🚫 **Hallucination Prevention**: Agent admits when it doesn't have the information

## 🎯 What is Agentic RAG?

Traditional RAG systems retrieve documents and pass them to a language model. **Agentic RAG** goes further by:
- Giving the agent **autonomous control** over when and how to search
- Allowing the agent to **decide** which retrieval tools to use
- Enabling **multi-step reasoning** with retrieved information
- Supporting **iterative refinement** of search queries

## 🏗️ Architecture

```
User Question → Agent Analysis → File Search Tool → Vector Store → Document Chunks
                                        ↓
                                Context-Aware Response
```

## 🔧 Prerequisites

**Required NuGet Packages:**
```bash
dotnet add package Microsoft.AgentFramework.Azure.AI
dotnet add package Azure.Identity
```

**Environment Variables:**
```env
AZURE_AI_FOUNDRY_MODEL=gpt-4o-mini
AZURE_AI_FOUNDRY_PROJECT_ENDPOINT=https://your-foundry-project.services.ai.azure.com/api/projects/yourProject
```

**Document Requirements:**
- Place your document (e.g., `document.md`) in the code_samples folder
- Supported formats: .md, .txt, .pdf, .docx

## 🚀 What This Notebook Does

1. **Upload Document**: Uploads a document to Azure AI Foundry
2. **Create Vector Store**: Generates embeddings for semantic search
3. **Configure RAG Agent**: Sets up agent with file search capabilities
4. **Query & Stream**: Asks questions and streams responses in real-time

Let's build an intelligent RAG agent! 🌟

In [ ]:
// 📦 Install Required NuGet Packages
#r "nuget: Microsoft.AgentFramework.Azure.AI"
#r "nuget: Azure.Identity"

In [ ]:
// 📦 Import Required Namespaces
using System;
using System.Threading.Tasks;
using System.Collections.Generic;

// Azure authentication
using Azure.Identity;

// Agent Framework components
using Microsoft.AgentFramework;
using Microsoft.AgentFramework.Azure;

In [ ]:
// 🔧 Load Environment Configuration
var modelDeploymentName = Environment.GetEnvironmentVariable("AZURE_AI_FOUNDRY_MODEL");
var projectEndpoint = Environment.GetEnvironmentVariable("AZURE_AI_FOUNDRY_PROJECT_ENDPOINT");

Console.WriteLine($"Model: {modelDeploymentName}");
Console.WriteLine($"Endpoint: {projectEndpoint}");

In [ ]:
// 📁 Vector Store Creation Helper Method
// Creates a vector store with uploaded documents for semantic search

async Task<(string fileId, HostedVectorStoreContent vectorStore)> CreateVectorStoreAsync(AzureAIAgentClient client)
{
    // Upload document to Azure AI Foundry
    var filePath = "./document.md";
    var file = await client.ProjectClient.Agents.Files.UploadAndPollAsync(
        filePath: filePath,
        purpose: "assistants"
    );
    Console.WriteLine($"Uploaded file, file ID: {file.Id}");

    // Create vector store with embeddings
    var vectorStore = await client.ProjectClient.Agents.VectorStores.CreateAndPollAsync(
        fileIds: new[] { file.Id },
        name: "graph_knowledge_base"
    );
    Console.WriteLine($"Created vector store, ID: {vectorStore.Id}");

    return (file.Id, new HostedVectorStoreContent(vectorStoreId: vectorStore.Id));
}

In [ ]:
// 🤖 Create RAG Agent with Azure AI Foundry
// Sets up agent with file search capabilities for document-grounded responses

var credential = new AzureCliCredential();

using var chatClient = new AzureAIAgentClient(
    credential: credential,
    modelDeploymentName: modelDeploymentName,
    projectEndpoint: new Uri(projectEndpoint)
);

// Create vector store and upload documents
var (fileId, vectorStore) = await CreateVectorStoreAsync(chatClient);

// Configure file search tool
var fileSearch = new HostedFileSearchTool(inputs: vectorStore);

// Create agent with RAG capabilities
var agent = chatClient.CreateAgent(
    name: "CSharpRAGDemo",
    instructions: @"
        You are an AI assistant designed to answer user questions using only the information retrieved from the provided document(s).

        - If a user's question cannot be answered using the retrieved context, **you must clearly respond**: 
        ""I'm sorry, but the uploaded document does not contain the necessary information to answer that question.""
        - Do not answer from general knowledge or reasoning. Do not make assumptions or generate hypothetical explanations.
        - Do not provide definitions, tutorials, or commentary that is not explicitly grounded in the content of the uploaded file(s).
        - If a user asks a question like ""What is a Neural Network?"", and this is not discussed in the uploaded document, respond as instructed above.
        - For questions that do have relevant content in the document (e.g., Contoso's travel insurance coverage), respond accurately, and cite the document explicitly.

        You must behave as if you have no external knowledge beyond what is retrieved from the uploaded document.
    ",
    tools: new[] { fileSearch },  // Tools available to the agent
    toolChoice: "auto"  // Let the agent decide when to use tools
);

Console.WriteLine("✅ Agent created. You can now ask questions about the uploaded document.");

In [ ]:
// 🚀 Query the RAG Agent with Streaming
// Ask questions and receive real-time streaming responses

var query = "Can you explain Contoso's travel insurance coverage?";
Console.WriteLine($"\n📝 Query: {query}\n");
Console.WriteLine("🤖 Response:\n");

await foreach (var chunk in agent.RunStreamAsync(
    query,
    toolResources: new Dictionary<string, object>
    {
        { "file_search", new { vector_store_ids = new[] { vectorStore.VectorStoreId } } }
    }
))
{
    if (!string.IsNullOrEmpty(chunk.Text))
    {
        Console.Write(chunk.Text);
    }
}

Console.WriteLine("\n\n✅ Response completed");

In [ ]:
// 🧪 Test with a Question Outside Document Scope
// Verify the agent refuses to answer questions not in the document

var outOfScopeQuery = "What is a Neural Network?";
Console.WriteLine($"\n📝 Query: {outOfScopeQuery}\n");
Console.WriteLine("🤖 Response:\n");

await foreach (var chunk in agent.RunStreamAsync(
    outOfScopeQuery,
    toolResources: new Dictionary<string, object>
    {
        { "file_search", new { vector_store_ids = new[] { vectorStore.VectorStoreId } } }
    }
))
{
    if (!string.IsNullOrEmpty(chunk.Text))
    {
        Console.Write(chunk.Text);
    }
}

Console.WriteLine("\n\n✅ Response completed");